# 🏥 Hyu PACS: In-House AI Bone Suppression Training Pipeline (v2.0)
### **Tahap 3 & 4: Sintesis DRR Multi-Sudut & Pelatihan Deep Learning Komersial**

Notebook ini melatih model Deep Learning **Bone Suppression Toraks** yang **100% Legal & Bebas Komersial** untuk disematkan langsung ke sistem **Hyu PACS**.

---
### 📋 Pembaruan Presisi Fisika Radiologi (v2.0):
1. **Shared Normalization (Pencegah Error Normalisasi Terpisah):** Nilai min-max ($P_{0.5}, P_{99.5}$) dihitung **hanya dari citra rontgen penuh ($X$)**, dan diskalakan identik ke citra soft-tissue ($Y$). Ini menjamin selisih $X - Y$ bernilai persis $0.0$ di seluruh jaringan lunak (paru, jantung, diafragma) dan hanya menyisakan tulang iga & klavikula murni.
2. **Segmentasi Tulang Komplit & Dilasi 2D:** Ambang batas $120\text{ HU}$ menjangkau trabekula (sumsum tulang) ditambah dilasi morfologis irisan aksial untuk mengeliminasi efek parsial volume di tepi korteks tulang iga.
3. **Multi-Angle Projections (140 Pasangan Data dari 10 CT):** Proyeksi sinar-X virtual diputar pada 7 sudut rotasi aksial ($-6^\circ, -4^\circ, -2^\circ, 0^\circ, +2^\circ, +4^\circ, +6^\circ$) ditambah flip horizontal.
4. **Supervisi Ganda (Dual Loss):** $\mathcal{L} = \mathcal{L}_{1,\text{bone}} + 0.8\cdot\mathcal{L}_{1,\text{soft}} + 0.3\cdot\mathcal{L}_{\text{grad}}$ (Mencegah blur nodul & pembuluh darah paru).
5. **Output Biner TorchScript:** `bone_suppression_commercial.ts` (Langsung jalan di Hyu PACS).

## 🛠️ Langkah 1: Periksa GPU & Pasang Pustaka Pendukung

In [ ]:
# Periksa ketersediaan GPU di Google Colab
!nvidia-smi

# Pasang pustaka medis resmi
!pip install -q tcia-utils pydicom SimpleITK monai opencv-python-headless matplotlib scipy

## 📂 Langkah 2: Hubungkan Google Drive (Menyimpan Checkpoint)

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/HyuPACS_BoneSuppression/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"[+] Folder checkpoint Google Drive siap: {CHECKPOINT_DIR}")

## 📥 Langkah 3: Unduh Otomatis Dataset CT Toraks LIDC-IDRI (TCIA)
Jika Anda sudah mengunduhnya sebelumnya, tahap ini akan otomatis mendeteksi dan melewatinya.

In [ ]:
import glob
import os
from tcia_utils import nbia

LIDC_RAW_DIR = '/content/lidc_raw'
os.makedirs(LIDC_RAW_DIR, exist_ok=True)

existing_patients = glob.glob(f"{LIDC_RAW_DIR}/patient_*")
if len(existing_patients) >= 10:
    print(f"[+] Data CT 10 pasien sudah lengkap tersedia di {LIDC_RAW_DIR}! Melewati unduhan TCIA.")
else:
    print("[*] Mengambil metadata seri CT toraks resolusi tinggi dari TCIA...")
    series_df = nbia.getSeries(
        collection="LIDC-IDRI",
        modality="CT",
        format="df"
    )
    target_series = series_df[series_df['ImageCount'] >= 120].head(10)
    series_uids = target_series['SeriesInstanceUID'].tolist()
    print(f"[+] Ditemukan {len(series_uids)} volume CT toraks berkualitas tinggi.")

    for i, s_uid in enumerate(series_uids, 1):
        patient_path = os.path.join(LIDC_RAW_DIR, f"patient_{i}")
        if not os.path.exists(patient_path) or len(os.listdir(patient_path)) == 0:
            os.makedirs(patient_path, exist_ok=True)
            print(f"    [{i}/{len(series_uids)}] Mengunduh Volume CT {s_uid[:20]}...")
            nbia.downloadSeries([{"SeriesInstanceUID": s_uid}], input_type="list", path=patient_path)
        else:
            print(f"    [{i}/{len(series_uids)}] Pasien {i} sudah ada di disk.")

    print("[+] Seluruh volume CT mentah siap di /content/lidc_raw!")

## 🔬 Langkah 4: Generator DRR v2.0 (Fisika Radiologi & Shared Normalization)
Membuat 140 pasangan citra beresolusi tinggi (1024x1024) dengan dekomposisi anatomis presisi.

In [ ]:
import glob
import os
import cv2
import pydicom
import numpy as np

def hu_to_attenuation(volume_hu):
    mu_water = 0.02  # mm^-1 pada rentang energi rontgen diagnostik (~70 keV)
    return np.maximum(0.0, mu_water * (1.0 + (volume_hu / 1000.0)))

def generate_multiangle_drr_pairs(dicom_patient_dir, output_size=(1024, 1024)):
    dcm_files = glob.glob(os.path.join(dicom_patient_dir, "**", "*.dcm"), recursive=True)
    if not dcm_files:
        return []
    
    slices = []
    for f in dcm_files:
        try:
            ds = pydicom.dcmread(f)
            if hasattr(ds, 'ImagePositionPatient') and hasattr(ds, 'pixel_array'):
                slices.append(ds)
        except Exception:
            continue
    if len(slices) < 30:
        return []
        
    # Urutkan irisan dari inferior ke superior (Z-axis)
    slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))
    vol = np.zeros((len(slices), slices[0].Rows, slices[0].Columns), dtype=np.float32)
    for idx, s in enumerate(slices):
        slope = float(getattr(s, 'RescaleSlope', 1.0))
        intercept = float(getattr(s, 'RescaleIntercept', 0.0))
        vol[idx] = s.pixel_array.astype(np.float32) * slope + intercept

    # 1. Dekomposisi Tulang: Threshold 120 HU mencakup trabekula & korteks
    bone_mask = vol > 120.0
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    bone_mask_dilated = np.zeros_like(bone_mask, dtype=bool)
    for z in range(vol.shape[0]):
        m = bone_mask[z].astype(np.uint8)
        if m.any():
            bone_mask_dilated[z] = cv2.dilate(m, kernel, iterations=1).astype(bool)

    vol_soft = vol.copy()
    vol_soft[bone_mask_dilated] = 35.0  # Rata-rata densitas otot & mediastinum

    # 2. Multi-Sudut Proyeksi (-6 s/d +6 derajat rotasi aksial)
    angles = [-6.0, -4.0, -2.0, 0.0, 2.0, 4.0, 6.0]
    h, w = vol.shape[1], vol.shape[2]
    center = (w / 2.0, h / 2.0)
    results = []

    for ang in angles:
        if abs(ang) > 1e-3:
            M = cv2.getRotationMatrix2D(center, ang, 1.0)
            vol_rot = np.empty_like(vol)
            vol_soft_rot = np.empty_like(vol_soft)
            for z in range(vol.shape[0]):
                vol_rot[z] = cv2.warpAffine(vol[z], M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
                vol_soft_rot[z] = cv2.warpAffine(vol_soft[z], M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        else:
            vol_rot = vol
            vol_soft_rot = vol_soft

        # Forward Ray-Summation (Hukum Beer-Lambert)
        mu_full = hu_to_attenuation(vol_rot)
        mu_soft = hu_to_attenuation(vol_soft_rot)
        proj_full = np.flipud(np.sum(mu_full, axis=1))
        proj_soft = np.flipud(np.sum(mu_soft, axis=1))

        # SHARED NORMALIZATION: Dihitung HANYA dari proj_full
        p1 = float(np.percentile(proj_full, 0.5))
        p99 = float(np.percentile(proj_full, 99.5))
        scale = max(p99 - p1, 1e-6)

        full_norm = np.clip((proj_full - p1) / scale, 0.0, 1.0).astype(np.float32)
        soft_norm = np.clip((proj_soft - p1) / scale, 0.0, 1.0).astype(np.float32)

        full_res = cv2.resize(full_norm, output_size, interpolation=cv2.INTER_AREA)
        soft_res = cv2.resize(soft_norm, output_size, interpolation=cv2.INTER_AREA)

        # Simpan orientasi asli
        suffix = f"ang_{int(ang):+02d}"
        results.append((suffix, full_res, soft_res))

        # Simpan variasi flip horizontal (kiri-kanan anatomis)
        results.append((f"{suffix}_flip", np.fliplr(full_res), np.fliplr(soft_res)))

    return results

# Eksekusi Pembuatan Dataset Pasangan
PAIRED_DIR = '/content/paired_dataset'
os.makedirs(f"{PAIRED_DIR}/x_full", exist_ok=True)
os.makedirs(f"{PAIRED_DIR}/y_soft", exist_ok=True)

patient_folders = sorted(glob.glob(f"{LIDC_RAW_DIR}/patient_*"))
count = 0
print(f"[*] Memulai sintesis DRR v2.0 dari {len(patient_folders)} subjek CT...")

for p_dir in patient_folders:
    p_name = os.path.basename(p_dir)
    pairs = generate_multiangle_drr_pairs(p_dir)
    for tag, x_img, y_img in pairs:
        fname = f"{p_name}_{tag}.png"
        cv2.imwrite(f"{PAIRED_DIR}/x_full/{fname}", (x_img * 65535).astype(np.uint16))
        cv2.imwrite(f"{PAIRED_DIR}/y_soft/{fname}", (y_img * 65535).astype(np.uint16))
        count += 1
    print(f"    [+] Selesai: {p_name} -> {len(pairs)} pasangan DRR.")

print(f"[+] Total pasangan citra latih presisi tinggi berhasil dibuat: {count}")

## 🖼️ Langkah 5: Inspeksi Visual Bukti Pemisahan Tulang (X - Y)
Panel ke-3 harus memperlihatkan **kerangka tulang iga & klavikula yang menyala tajam**, sedangkan paru-paru dan jantung **berwarna hitam pekat (0.0)**.

In [ ]:
import matplotlib.pyplot as plt

sample_files = sorted(glob.glob(f"{PAIRED_DIR}/x_full/*.png"))
if sample_files:
    sample_name = os.path.basename(sample_files[0])
    img_x = cv2.imread(f"{PAIRED_DIR}/x_full/{sample_name}", cv2.IMREAD_UNCHANGED).astype(np.float32) / 65535.0
    img_y = cv2.imread(f"{PAIRED_DIR}/y_soft/{sample_name}", cv2.IMREAD_UNCHANGED).astype(np.float32) / 65535.0
    bone_gt = np.clip(img_x - img_y, 0.0, 1.0)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].set_title("1. Input: Rontgen Penuh (Ada Iga)", fontsize=12, fontweight='bold')
    axes[0].imshow(img_x, cmap='gray')
    axes[0].axis('off')

    axes[1].set_title("2. Target: Soft-Tissue Murni (Tanpa Iga)", fontsize=12, fontweight='bold')
    axes[1].imshow(img_y, cmap='gray')
    axes[1].axis('off')

    axes[2].set_title("3. Ground Truth Tulang (X - Y) [Murni Tulang! ]", fontsize=12, fontweight='bold', color='crimson')
    im3 = axes[2].imshow(bone_gt, cmap='inferno')
    plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

## 🧠 Langkah 6: Arsitektur Model Deep Learning (MONAI Residual UNet)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from monai.networks.nets import UNet

class DRRPairDataset(Dataset):
    def __init__(self, x_paths, y_paths):
        self.x_paths = x_paths
        self.y_paths = y_paths

    def __len__(self):
        return len(self.x_paths)

    def __getitem__(self, idx):
        x = cv2.imread(self.x_paths[idx], cv2.IMREAD_UNCHANGED).astype(np.float32) / 65535.0
        y = cv2.imread(self.y_paths[idx], cv2.IMREAD_UNCHANGED).astype(np.float32) / 65535.0
        bone_target = np.clip(x - y, 0.0, 1.0)

        x_t = torch.from_numpy(x)[None, ...]
        bone_t = torch.from_numpy(bone_target)[None, ...]
        soft_t = torch.from_numpy(y)[None, ...]
        return x_t, bone_t, soft_t

class CommercialBoneSuppressionNet(nn.Module):
    """Wrapper TorchScript untuk memastikan output selalu terikat pada rentang anatomis [0, 1]."""
    def __init__(self, unet):
        super().__init__()
        self.unet = unet
        
    def forward(self, x):
        bone = self.unet(x)
        return torch.clamp(bone, 0.0, 1.0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
base_unet = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(32, 64, 128, 256, 512),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm="batch"
).to(device)

model = CommercialBoneSuppressionNet(base_unet).to(device)
print(f"[+] Model diinisialisasi di perangkat: {device}")
print(f"[+] Total Parameter Jaringan: {sum(p.numel() for p in model.parameters()):,}")

## ⚡ Langkah 7: Proses Pelatihan (Dual Supervision & Gradient Preservation)

In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler

all_x = sorted(glob.glob(f"{PAIRED_DIR}/x_full/*.png"))
all_y = sorted(glob.glob(f"{PAIRED_DIR}/y_soft/*.png"))

train_ds = DRRPairDataset(all_x, all_y)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, pin_memory=True)

optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-5)
scaler = GradScaler()

EPOCHS = 40
print(f"[*] Memulai pelatihan model Bone Suppression ({len(train_ds)} sampel, {EPOCHS} epoch)...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for x_b, target_bone, target_soft in train_loader:
        x_b = x_b.to(device)
        target_bone = target_bone.to(device)
        target_soft = target_soft.to(device)
        
        optimizer.zero_grad()
        with autocast():
            pred_bone = model(x_b)
            pred_soft = torch.clamp(x_b - pred_bone, 0.0, 1.0)
            
            # 1. Presisi Isolasi Tulang
            loss_bone = F.l1_loss(pred_bone, target_bone)
            
            # 2. Rekonstruksi Jaringan Lunak
            loss_soft = F.l1_loss(pred_soft, target_soft)
            
            # 3. Preservasi Gradien Struktur Paru (mencegah blur nodul & bronkus)
            gx_pred = torch.abs(pred_soft[:, :, :, :-1] - pred_soft[:, :, :, 1:])
            gx_tgt = torch.abs(target_soft[:, :, :, :-1] - target_soft[:, :, :, 1:])
            gy_pred = torch.abs(pred_soft[:, :, :-1, :] - pred_soft[:, :, 1:, :])
            gy_tgt = torch.abs(target_soft[:, :, :-1, :] - target_soft[:, :, 1:, :])
            loss_grad = F.l1_loss(gx_pred, gx_tgt) + F.l1_loss(gy_pred, gy_tgt)
            
            loss = loss_bone + 0.8 * loss_soft + 0.3 * loss_grad
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        
    scheduler.step()
    avg_loss = running_loss / max(1, len(train_loader))
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] - Total Loss: {avg_loss:.5f} (lr: {scheduler.get_last_lr()[0]:.6f})")
    
    if epoch % 10 == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"bone_suppression_v2_epoch_{epoch}.pth")
        torch.save(model.state_dict(), ckpt_path)
        print(f"    -> Checkpoint tersimpan di Drive: {ckpt_path}")

print("[+] Pelatihan tuntas dengan sempurna!")

## 📦 Langkah 8: Export ke TorchScript (`bone_suppression_commercial.ts`)

In [ ]:
model.eval()
dummy_input = torch.rand(1, 1, 1024, 1024).to(device)

traced_model = torch.jit.trace(model, dummy_input)

output_ts_name = "bone_suppression_commercial.ts"
traced_model.save(output_ts_name)

drive_target = os.path.join(CHECKPOINT_DIR, output_ts_name)
traced_model.save(drive_target)

file_size_mb = os.path.getsize(output_ts_name) / (1024 * 1024)
print(f"[+] File Model Jadi Berhasil Dibuat: {output_ts_name} ({file_size_mb:.2f} MB)")
print(f"[+] Cadangan permanen tersimpan di Google Drive: {drive_target}")

## ⬇️ Langkah 9: Download File Model ke Laptop Anda

In [ ]:
from google.colab import files

print("[*] Mengunduh file 'bone_suppression_commercial.ts' ke laptop...")
files.download(output_ts_name)

print("""
=========================================================================
Langkah Selanjutnya di Laptop Anda:
1. Pindahkan file 'bone_suppression_commercial.ts' ke folder:
   d:\laragon\www\belajar-laravel\medical_engine\weights\
2. Buka Hyu PACS di browser: http://127.0.0.1:8000/scans/1
3. Pilih dropdown '[AI-01] Bone Suppression (In-House Commercial Engine)'
4. Klik '▶ Jalankan In-House AI Bone Suppression' -> Geser Tirai A/B!
=========================================================================
""")